In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Carrega os dados
orders = pd.read_csv(r'C:\Users\olist\olist_orders_dataset.csv')
payments = pd.read_csv(r'C:\Users\olist\olist_order_payments_dataset.csv')

# Agrupa pagamentos por pedido somando o valor total
total_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()
total_por_pedido.columns = ['order_id', 'valor_total']

# Filtra pedidos acima de R$ 150
pedidos_acima_150 = total_por_pedido[total_por_pedido['valor_total'] > 150]

# Junta com a tabela de pagamentos pra pegar o método
# Pega o método de pagamento predominante por pedido (o de maior valor)
metodo_por_pedido = payments.loc[
    payments.groupby('order_id')['payment_value'].idxmax()
][['order_id', 'payment_type']]

# Faz o merge
df = pedidos_acima_150.merge(metodo_por_pedido, on='order_id')

# Conta e calcula percentual
contagem = df['payment_type'].value_counts().reset_index()
contagem.columns = ['metodo', 'quantidade']
contagem['percentual'] = (contagem['quantidade'] / contagem['quantidade'].sum()) * 100

# Traduz os nomes
traducao = {
    'credit_card': 'Cartão de Crédito',
    'boleto': 'Boleto',
    'voucher': 'Voucher',
    'debit_card': 'Cartão de Débito',
    'not_defined': 'Não Definido'
}
contagem['metodo'] = contagem['metodo'].map(traducao).fillna(contagem['metodo'])

print("=== Método de Pagamento — Pedidos acima de R$ 150,00 ===\n")
for _, row in contagem.iterrows():
    print(f"{row['metodo']:<25} {row['quantidade']:>6} pedidos  ({row['percentual']:.1f}%)")

# ---- Gráfico ----
cores = ['#2ECC71', '#3498DB', '#E74C3C', '#F39C12', '#9B59B6']

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.barh(
    contagem['metodo'],
    contagem['percentual'],
    color=cores[:len(contagem)],
    edgecolor='white',
    height=0.5
)

# Anotação de % e quantidade em cada barra
for bar, (_, row) in zip(bars, contagem.iterrows()):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{row['percentual']:.1f}%  ({row['quantidade']:,} pedidos)",
        va='center',
        fontsize=10
    )

ax.set_title('Método de Pagamento em Pedidos acima de R$ 150,00', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Percentual (%)', fontsize=11)
ax.set_xlim(0, contagem['percentual'].max() * 1.35)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('metodo_pagamento.png', dpi=150)
plt.show()